# Step 1 — CrisisMMD Dataset Exploration

This notebook inspects the *actual* downloaded CrisisMMD files before any preprocessing assumptions are made. It does not hardcode column names — it discovers them from the real files placed in `data/raw/`.

**Before running:** download the annotation split from https://crisisnlp.qcri.org/crisismmd (`crisismmd_datasplit_all.zip`), unzip it, and place the contents under `data/raw/`.

In [ ]:
import sys
sys.path.insert(0, '..')
from pathlib import Path
import pandas as pd
import glob, json
from collections import defaultdict

RAW_DIR = Path('../data/raw')
SUPPORTED_EXTS = {'.csv', '.tsv', '.txt'}

def find_data_files():
    files = []
    for ext in SUPPORTED_EXTS:
        files.extend(glob.glob(str(RAW_DIR / '**' / f'*{ext}'), recursive=True))
    return sorted(set(files))

files = find_data_files()
print(f'Found {len(files)} file(s):')
for f in files:
    print(' -', Path(f).relative_to(RAW_DIR))

## Load and inspect each file

For each file: shape, columns, dtypes, a guessed text column, guessed label columns (low-cardinality object columns), missing values, and duplicates.

In [ ]:
def sniff_separator(path):
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        first_line = f.readline()
    return '\t' if first_line.count('\t') > first_line.count(',') else ','

def guess_text_column(df):
    candidates = {}
    for col in df.columns:
        if df[col].dtype == object:
            try:
                candidates[col] = df[col].astype(str).str.len().mean()
            except Exception:
                continue
    return max(candidates, key=candidates.get) if candidates else None

def guess_label_columns(df, text_col, max_unique=25):
    label_cols = []
    n = len(df)
    for col in df.columns:
        if col == text_col:
            continue
        nunique = df[col].nunique(dropna=True)
        if 1 < nunique <= max_unique and nunique < n * 0.5:
            label_cols.append(col)
    return label_cols

dfs = {}
for f in files:
    p = Path(f)
    sep = sniff_separator(p)
    df = pd.read_csv(p, sep=sep, engine='python', on_bad_lines='warn')
    dfs[str(p.relative_to(RAW_DIR))] = df
    text_col = guess_text_column(df)
    label_cols = guess_label_columns(df, text_col)
    print('='*80)
    print('FILE:', p.relative_to(RAW_DIR))
    print('Shape:', df.shape)
    print('Columns:', list(df.columns))
    print('Guessed text column:', text_col)
    print('Guessed label columns:', label_cols)
    print('Missing values:\n', df.isna().sum()[df.isna().sum() > 0])
    print('Duplicate rows:', df.duplicated().sum())
    for col in label_cols:
        print(f'-- {col} distribution --')
        print(df[col].value_counts(dropna=False))
    print()

## Cross-file overlap check

CrisisMMD ships separate files per task (informative / humanitarian / damage) and per split (train/dev/test). The same tweet can legitimately appear in multiple *task* files, but must **not** appear in both a train file and a test file for the same task — that would be leakage.

In [ ]:
# Fill in once real text/id columns are confirmed above.
# Example pattern (edit column names to match what you saw printed):
# text_index = defaultdict(list)
# for fname, df in dfs.items():
#     for t in df['<confirmed_text_column>'].astype(str):
#         text_index[t].append(fname)
# overlap = {t: fs for t, fs in text_index.items() if len(set(fs)) > 1}
# print(f'{len(overlap)} text values appear in more than one file')
pass

## Save exploration stats

Persist findings so Step 2 (clean dataframe + splits) can be built from confirmed facts rather than assumptions.

In [ ]:
summary = {fname: {'shape': list(df.shape), 'columns': list(df.columns)} for fname, df in dfs.items()}
out_path = Path('../data/processed/step1_exploration_stats.json')
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('Saved to', out_path)